Now let's see directly how a xgb will perform

In [51]:
import pandas as pd
import numpy as np
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from pathlib import Path

In [33]:
from utils import score, compute_costs

In [34]:
x_train = pd.read_csv(Path.cwd().parent / "data/german_credit_train.csv")
x_test = pd.read_csv(Path.cwd().parent / "data/german_credit_test.csv")

In [35]:
x_test.head()

,Id,CheckingStatus,LoanDuration,CreditHistory,LoanPurpose,LoanAmount,ExistingSavings,EmploymentDuration,InstallmentPercent,Sex,...,CurrentResidenceDuration,OwnsProperty,Age,InstallmentPlans,Housing,ExistingCreditsCount,Job,Dependents,Telephone,ForeignWorker
0,0,no_checking,9,prior_payments_delayed,car_new,1032,100_to_500,4_to_7,3,male,...,4,savings_insurance,41,none,own,1,management_self-employed,1,none,yes
1,1,less_0,5,all_credits_paid_back,car_new,1523,less_100,unemployed,2,female,...,2,real_estate,19,none,rent,1,management_self-employed,1,none,yes
2,2,no_checking,39,prior_payments_delayed,repairs,7150,500_to_1000,4_to_7,3,male,...,4,unknown,52,none,own,2,skilled,1,yes,yes
3,3,0_to_200,15,prior_payments_delayed,furniture,250,500_to_1000,4_to_7,3,male,...,2,savings_insurance,24,none,own,2,skilled,2,yes,yes
4,4,0_to_200,16,prior_payments_delayed,car_new,5551,100_to_500,1_to_4,3,male,...,3,car_other,34,none,rent,2,management_self-employed,1,none,yes


In [36]:
x_train.head()

,CheckingStatus,LoanDuration,CreditHistory,LoanPurpose,LoanAmount,ExistingSavings,EmploymentDuration,InstallmentPercent,Sex,OthersOnLoan,...,OwnsProperty,Age,InstallmentPlans,Housing,ExistingCreditsCount,Job,Dependents,Telephone,ForeignWorker,Risk
0,0_to_200,31,credits_paid_to_date,other,1889,100_to_500,less_1,3,female,none,...,savings_insurance,32,none,own,1,skilled,1,none,yes,No Risk
1,less_0,18,credits_paid_to_date,car_new,462,less_100,1_to_4,2,female,none,...,savings_insurance,37,stores,own,2,skilled,1,none,yes,No Risk
2,less_0,15,prior_payments_delayed,furniture,250,less_100,1_to_4,2,male,none,...,real_estate,28,none,own,2,skilled,1,yes,no,No Risk
3,0_to_200,28,credits_paid_to_date,retraining,3693,less_100,greater_7,3,male,none,...,savings_insurance,32,none,own,1,skilled,1,none,yes,No Risk
4,no_checking,28,prior_payments_delayed,education,6235,500_to_1000,greater_7,3,male,none,...,unknown,57,none,own,2,skilled,1,none,yes,Risk


In [37]:
x_test.drop(columns='Id')


,CheckingStatus,LoanDuration,CreditHistory,LoanPurpose,LoanAmount,ExistingSavings,EmploymentDuration,InstallmentPercent,Sex,OthersOnLoan,CurrentResidenceDuration,OwnsProperty,Age,InstallmentPlans,Housing,ExistingCreditsCount,Job,Dependents,Telephone,ForeignWorker
0,no_checking,9,prior_payments_delayed,car_new,1032,100_to_500,4_to_7,3,male,none,4,savings_insurance,41,none,own,1,management_self-employed,1,none,yes
1,less_0,5,all_credits_paid_back,car_new,1523,less_100,unemployed,2,female,none,2,real_estate,19,none,rent,1,management_self-employed,1,none,yes
2,no_checking,39,prior_payments_delayed,repairs,7150,500_to_1000,4_to_7,3,male,co-applicant,4,unknown,52,none,own,2,skilled,1,yes,yes
3,0_to_200,15,prior_payments_delayed,furniture,250,500_to_1000,4_to_7,3,male,none,2,savings_insurance,24,none,own,2,skilled,2,yes,yes
4,0_to_200,16,prior_payments_delayed,car_new,5551,100_to_500,1_to_4,3,male,none,3,car_other,34,none,rent,2,management_self-employed,1,none,yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
996,no_checking,38,credits_paid_to_date,appliances,5308,100_to_500,4_to_7,3,male,none,3,savings_insurance,31,none,own,1,skilled,1,none,yes
997,less_0,31,credits_paid_to_date,retraining,1997,less_100,1_to_4,3,male,none,3,savings_insurance,31,none,own,2,skilled,2,none,yes
998,less_0,20,prior_payments_delayed,radio_tv,1155,greater_1000,1_to_4,3,male,none,3,savings_insurance,33,none,rent,2,skilled,1,yes,yes
999,less_0,4,credits_paid_to_date,car_new,250,less_100,unemployed,1,female,none,1,real_estate,23,none,rent,1,skilled,1,none,yes


In [38]:
y_train = x_train[['LoanAmount', 'Risk']]
x_train = x_train.drop(columns='Risk')

In [39]:
numerical_columns = x_train.select_dtypes(include = np.number).columns.tolist()
categorical_columns = x_train.select_dtypes(exclude=np.number).columns.tolist()

In [40]:
X_valid, X_predictions, Y_valid, Y_predictions = train_test_split(x_train, y_train)

In [41]:
Y_valid = Y_valid.drop(columns = 'LoanAmount') # Otherwise the model won't know what to aim for

### XGB with OneHotEncoding

In [42]:
preprocessor = preprocessor = ColumnTransformer(
    [
        ("num", StandardScaler(), numerical_columns),
        ("cat", OneHotEncoder(), categorical_columns),
        ]
)

In [43]:
model = HistGradientBoostingClassifier()


pipe = make_pipeline(preprocessor, model)
pipe.fit(X_valid, Y_valid)

y_pred = pipe.predict(X_predictions)

# let's have the submission to the right format
submission = pd.DataFrame(data = y_pred)
submission = submission.reset_index()
submission.columns = ["ID", "Risk"]


/Applications/anaconda3/lib/python3.12/site-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


In [44]:
score (Y_predictions, submission, "XGB")

-53.72226600000008

### XGB with Ordinal Encoding

In [52]:
preprocessor = preprocessor = ColumnTransformer(
    [
        ("num", StandardScaler(), numerical_columns),
        ("cat", OrdinalEncoder(), categorical_columns),
        ]
)

In [53]:
model = HistGradientBoostingClassifier()


pipe = make_pipeline(preprocessor, model)
pipe.fit(X_valid, Y_valid)

y_pred = pipe.predict(X_predictions)

# let's have the submission to the right format
submission = pd.DataFrame(data = y_pred)
submission = submission.reset_index()
submission.columns = ["ID", "Risk"]


/Applications/anaconda3/lib/python3.12/site-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


In [54]:
score (Y_predictions, submission, "XGB")

-55.48773600000008

Ordinal Encoding works better because it gives automatically an importance via the number to categorical features

Now let's submit a prediction

In [45]:
y_train = y_train.drop(columns = ["LoanAmount"])

In [55]:
pipe.fit(x_train, y_train)

/Applications/anaconda3/lib/python3.12/site-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


Pipeline(steps=[('columntransformer',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['LoanDuration', 'LoanAmount',
                                                   'InstallmentPercent',
                                                   'CurrentResidenceDuration',
                                                   'Age',
                                                   'ExistingCreditsCount',
                                                   'Dependents']),
                                                 ('cat', OrdinalEncoder(),
                                                  ['CheckingStatus',
                                                   'CreditHistory',
                                                   'LoanPurpose',
                                                   'ExistingSavings',
                                                   'EmploymentDuration', 'Sex',
                                                   'OthersOnLoan',
                                                   'OwnsProperty',
                                                   'InstallmentPlans',
                                                   'Housing', 'Job',
                                                   'Telephone',
                                                   'ForeignWorker'])])),
                ('histgradientboostingclassifier',
                 HistGradientBoostingClassifier())])

In [56]:
y_pred = pipe.predict(x_test)

In [57]:
submission = pd.DataFrame(data = y_pred)
submission = submission.reset_index()
submission.columns = ["Id", "Risk"]

In [58]:
submission.to_csv('submission_xgb.csv')